## Diseño de redes

In [1]:
import pandas as pd

data = pd.read_excel("data/Tsp-1.xlsx")
data.columns
print(data.iloc[0,:])

colonia          U H FOVISSSTE ESTADI
Calle                          LISBOA
# de casa                          28
Código postal                   44307
Unnamed: 4                Guadalajara
Name: 0, dtype: object


In [2]:
import pandas as pd
from geopy.geocoders import ArcGIS
import time

def pre_procesar_direcciones(ruta_entrada, ruta_salida, cantidad_a_procesar=None):
    df = pd.read_excel(ruta_entrada)
    
    if cantidad_a_procesar:
        df = df.head(cantidad_a_procesar).copy() 
    
    geolocator = ArcGIS()

    lats = []
    lons = []
    encontrados = []

    print(f" Procesando {len(df)} direcciones. ")

    for index, row in df.iterrows():
        calle = str(row['Calle']).replace('#', '').strip()
        numero = str(row['# de casa']).replace('#', '').strip()
        colonia = str(row['colonia']).strip()
        ciudad = str(row['Unnamed: 4']).strip()
        
        direccion = f"{calle} {numero}, {colonia}, {ciudad}, Mexico"
        
        try:
            location = geolocator.geocode(direccion, timeout=10)
            if location:
                lats.append(location.latitude)
                lons.append(location.longitude)
                encontrados.append(True)
                print(f"    {index}: OK")
            else:
                lats.append(None)
                lons.append(None)
                encontrados.append(False)
                print(f"    {index}: No encontrado")
        except:
            lats.append(None)
            lons.append(None)
            encontrados.append(False)
            print(f"    {index}: Error de conexión")

        time.sleep(0.7)

    df['latitud'] = lats
    df['longitud'] = lons
    df["id_original_excel"] = df.index + 1
    
    df_final = df[df['latitud'].notnull()]
    
    df_final.to_csv(ruta_salida, index=False)
    print(f"\n Guardado exitosamente en '{ruta_salida}'. Nodos válidos: {len(df_final)}")

 

In [ ]:
pre_procesar_direcciones('Tsp-1.xlsx', 'direcciones_lat_lon.csv')

 Procesando 289 direcciones. 
    0: OK
    1: OK
    2: OK
    3: OK
    4: OK
    5: OK
    6: OK
    7: OK
    8: OK
    9: OK
    10: OK
    11: OK
    12: OK
    13: OK
    14: OK
    15: OK
    16: OK
    17: OK
    18: OK
    19: OK
    20: OK
    21: OK
    22: OK
    23: OK
    24: OK
    25: OK
    26: OK
    27: OK
    28: OK
    29: OK
    30: OK
    31: OK
    32: OK
    33: OK
    34: OK
    35: OK
    36: OK
    37: OK
    38: OK
    39: OK
    40: OK
    41: OK
    42: OK
    43: OK
    44: OK
    45: OK
    46: OK
    47: OK
    48: OK
    49: OK
    50: OK
    51: OK
    52: OK
    53: OK
    54: OK
    55: OK
    56: OK
    57: OK
    58: OK
    59: OK
    60: OK
    61: OK
    62: OK
    63: OK
    64: OK
    65: OK
    66: OK
    67: OK
    68: OK
    69: OK
    70: OK
    71: OK
    72: OK
    73: OK
    74: OK
    75: OK
    76: OK
    77: OK
    78: OK
    79: OK
    80: OK
    81: OK
    82: OK
    83: OK
    84: OK
    85: OK
    86: OK
    87: OK
    88: OK
 

In [ ]:
import folium
import pandas as pd

data = pd.read_csv('direcciones_lat_lon_limpio.csv')
def visualizar_nodos(direcciones_info):
    for row in direcciones_info.itertuples():
        folium.Marker(
            location=[row.latitud, row.longitud],
            popup=f"ID: {row.id_original_excel}\n{row.Calle}, {row.colonia}",
            icon=folium.Icon(color='blue', icon='info-sign')
        ).add_to(mapa)
    return mapa
mapa = folium.Map(location=[data['latitud'].mean(), data['longitud'].mean()], zoom_start=12)
m = visualizar_nodos(data)
m.save("mapa_direcciones.html")

## Análisis de nodos atípicos (polizones)

In [ ]:
from geopy.distance import geodesic
import numpy as np

def analizar_nodo_polizon(df):
    """
    Analiza todos los nodos y encuentra el que está más lejos en promedio de los demás.
    Un nodo muy alejado podría ser de otra ciudad.
    """
    n = len(df)
    distancias_promedio = []
    
    for idx, row in df.iterrows():
        lat1, lon1 = row['latitud'], row['longitud']
        distancias = []
        
        # Calcular distancia a todos los demás nodos
        for idx2, row2 in df.iterrows():
            if idx != idx2:
                lat2, lon2 = row2['latitud'], row2['longitud']
                dist = geodesic((lat1, lon1), (lat2, lon2)).kilometers
                distancias.append(dist)
        
        promedio = np.mean(distancias)
        distancias_promedio.append({
            'idx': idx,
            'id_original': row.get('id_original_excel', 'N/A'),
            'calle': row.get('Calle', ''),
            'numero': row.get('# de casa', ''),
            'colonia': row.get('colonia', ''),
            'latitud': lat1,
            'longitud': lon1,
            'distancia_promedio': promedio,
            'distancia_max': max(distancias),
            'distancia_min': min(distancias)
        })
    
    # Ordenar por distancia promedio (mayor a menor)
    distancias_promedio.sort(key=lambda x: x['distancia_promedio'], reverse=True)
    
    return distancias_promedio

# Analizar
resultados = analizar_nodo_polizon(data)

print("=" * 80)
print("TOP 10 NODOS MÁS ALEJADOS EN PROMEDIO:")
print("=" * 80)
for i, nodo in enumerate(resultados[:10], 1):
    print(f"\n{i}. Índice: {nodo['idx']} | ID Original: {nodo['id_original']}")
    print(f"   Dirección: {nodo['calle']} {nodo['numero']}, {nodo['colonia']}")
    print(f"   Coords: ({nodo['latitud']:.6f}, {nodo['longitud']:.6f})")
    print(f"   Distancia promedio: {nodo['distancia_promedio']:.2f} km")
    print(f"   Distancia mínima: {nodo['distancia_min']:.2f} km")
    print(f"   Distancia máxima: {nodo['distancia_max']:.2f} km")

print("\n" + "=" * 80)
print("TOP 2 NODOS SOSPECHOSOS (polizones potenciales):")
print("=" * 80)
polizones = resultados[:2]
for i, polizon in enumerate(polizones, 1):
    print(f"\nPOLIZÓN #{i}:")
    print(f"  Índice: {polizon['idx']}")
    print(f"  ID Original Excel: {polizon['id_original']}")
    print(f"  Dirección completa: {polizon['calle']} {polizon['numero']}, {polizon['colonia']}")
    print(f"  Coordenadas: ({polizon['latitud']:.6f}, {polizon['longitud']:.6f})")
    print(f"  Distancia promedio a otros nodos: {polizon['distancia_promedio']:.2f} km")
    print(f"  Distancia al nodo más cercano: {polizon['distancia_min']:.2f} km")

# Calcular estadísticas generales
distancias_promedio_todas = [x['distancia_promedio'] for x in resultados]
media = np.mean(distancias_promedio_todas)
desviacion = np.std(distancias_promedio_todas)
percentil_95 = np.percentile(distancias_promedio_todas, 95)

print("\n" + "=" * 80)
print("ESTADÍSTICAS GENERALES:")
print("=" * 80)
print(f"Media de distancias promedio: {media:.2f} km")
print(f"Desviación estándar: {desviacion:.2f} km")
print(f"Percentil 95: {percentil_95:.2f} km")
for i, polizon in enumerate(polizones, 1):
    print(f"Polizón #{i} está a {(polizon['distancia_promedio'] - media) / desviacion:.2f} desviaciones estándar de la media")


In [ ]:
# Visualizar los 2 polizones en el mapa
import folium

polizones = resultados[:2]

# Crear mapa centrado en el promedio de todos los nodos
mapa_polizon = folium.Map(
    location=[data['latitud'].mean(), data['longitud'].mean()], 
    zoom_start=11
)

# Crear conjunto de índices de polizones para excluir
indices_polizones = {p['idx'] for p in polizones}

# Agregar todos los nodos normales (en azul)
for idx, row in data.iterrows():
    if idx not in indices_polizones:
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=3,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=f"{row.get('Calle', '')} {row.get('# de casa', '')}"
        ).add_to(mapa_polizon)

# Agregar los 2 polizones en ROJO
colores = ['red', 'darkred']
for i, polizon in enumerate(polizones):
    folium.Marker(
        location=[polizon['latitud'], polizon['longitud']],
        popup=f"<b>POLIZÓN #{i+1}</b><br>{polizon['calle']} {polizon['numero']}<br>{polizon['colonia']}<br>Dist. promedio: {polizon['distancia_promedio']:.2f} km",
        icon=folium.Icon(color=colores[i], icon='warning-sign')
    ).add_to(mapa_polizon)
    
    folium.CircleMarker(
        location=[polizon['latitud'], polizon['longitud']],
        radius=15,
        color=colores[i],
        fill=True,
        fill_color=colores[i],
        fill_opacity=0.8
    ).add_to(mapa_polizon)

mapa_polizon.save("mapa_polizon_detectado.html")
print(f"Mapa guardado en: mapa_polizon_detectado.html")
print(f"Se detectaron {len(polizones)} polizones marcados en rojo")
mapa_polizon


In [ ]:
# Eliminar los 2 polizones del dataset y guardar versión limpia
polizones = resultados[:2]
indices_a_eliminar = [p['idx'] for p in polizones]

data_limpio = data[~data.index.isin(indices_a_eliminar)].copy()

print(f"Dataset original: {len(data)} nodos")
print(f"Dataset limpio: {len(data_limpio)} nodos")
print(f"Nodos eliminados: {len(indices_a_eliminar)}")
print()
for i, polizon in enumerate(polizones, 1):
    print(f"  Polizón #{i}: {polizon['calle']} {polizon['numero']}, {polizon['colonia']}")
    print(f"             Dist. promedio: {polizon['distancia_promedio']:.2f} km")

print(f"\nSe guardará como: direcciones_lat_lon_limpio_sin_polizon.csv")

# Guardar
data_limpio.to_csv('direcciones_lat_lon_limpio_sin_polizon.csv', index=False)
print("✓ Archivo guardado exitosamente")
